# 02 SQL metrics

> **Simulated case study.** Glowina Skincare is a fictional brand and all data is synthetic.

This notebook runs the SQL files in `sql/` against the clean tables from notebook 01 and reads the results. The queries use a SQLite database built in memory, so there is nothing to install beyond pandas.

Metric rules used everywhere:

- **Net revenue** counts only orders with status Completed.
- **CAC** is spend divided by new customers, where a new customer is credited to the campaign of their first order.
- **ROAS** uses platform-reported revenue, **net ROAS** uses net revenue.
- Orders without a campaign are organic repeat orders and are never counted as ad revenue.
- Revenue is not margin. No product cost exists in this data, so a campaign above 1.0x can still lose money.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

CLEAN = Path("../data/clean")
SQL = Path("../sql")
OUT = Path("../data/analytics")
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

con = sqlite3.connect(":memory:")
for name in ["campaigns", "daily_performance", "customers", "orders"]:
    pd.read_csv(CLEAN / f"{name}.csv").to_sql(name, con, index=False)
con.executescript((SQL / "01_views.sql").read_text())

def run(sql_file, export=None, edge_rows=8):
    """Run one SQL file, optionally save the result, and print it."""
    df = pd.read_sql((SQL / sql_file).read_text(), con)
    if export:
        df.to_csv(OUT / f"{export}.csv", index=False)
    if len(df) > 2 * edge_rows + 2:
        print(df.head(edge_rows).to_string(index=False))
        print(f"... {len(df) - 2 * edge_rows} more rows, full table in data/analytics/{export}.csv ...")
        print(df.tail(edge_rows).to_string(index=False))
    else:
        print(df.to_string(index=False))
    return df

# The view must agree with the raw tables before anything is built on it
check = pd.read_sql("""
SELECT (SELECT SUM(spend_idr) FROM daily_performance) AS spend_tables,
       (SELECT SUM(spend) FROM v_campaign_summary)     AS spend_view,
       (SELECT SUM(order_value_idr) FROM orders
         WHERE campaign_id IS NOT NULL AND status = 'Completed') AS net_tables,
       (SELECT SUM(net_revenue) FROM v_campaign_summary) AS net_view
""", con)
assert check.spend_tables[0] == check.spend_view[0] and check.net_tables[0] == check.net_view[0]
print(f"total spend IDR {check.spend_view[0]:,.0f} | total net revenue IDR {check.net_view[0]:,.0f}")
print(f"blended net ROAS {check.net_view[0] / check.spend_view[0]:.2f}x")

total spend IDR 1,417,489,500 | total net revenue IDR 2,521,033,000
blended net ROAS 1.78x

## 1. Channel performance

`sql/02_channel_performance.sql`: funnel metrics per channel, computed from summed totals.

In [2]:
channels = run("02_channel_performance.sql", export="channel_performance")

   channel  spend_idr  spend_share_pct  impressions  clicks  ctr_pct  cpc_idr  cvr_pct  new_customers  cac_idr  roas_reported  roas_net  non_completed_pct
Google Ads  317258000             22.4      9688206  133345     1.38   2379.0     2.18           2069 153339.0           2.99      2.81                6.2
  Meta Ads  470722200             33.2     13563864  144179     1.06   3265.0     2.63           2549 184669.0           2.47      2.29                7.5
TikTok Ads  352669600             24.9     17047155  267782     1.57   1317.0     0.65           1598 220694.0           1.04      0.94                9.8
Influencer  276839700             19.5      5003600   44180     0.88   6266.0     2.01            817 338849.0           0.88      0.81                8.2

Google Ads (2.81x) and Meta Ads (2.29x) return more than they cost. TikTok (0.94x) and Influencer (0.81x) return less than they cost on first-order revenue alone. TikTok has the lowest conversion rate (0.65%) and the highest share of cancelled or returned orders (9.8%). Influencer clicks cost about IDR 6,300 each, nearly twice a Meta click and almost five times a TikTok click.

Platform ROAS is higher than net ROAS in every channel because cancelled and returned orders still count as platform revenue.

## 2. Campaign scorecard

`sql/03_campaign_scorecard.sql`: every campaign ranked by net ROAS, with a tier. Scale is 3.0x or more, Keep is 1.5x to 3.0x, Fix is 1.0x to 1.5x and Cut is below 1.0x.

In [3]:
scorecard = run("03_campaign_scorecard.sql", export="campaign_scorecard")

campaign_id                                    campaign_name    channel        objective  spend_idr  net_revenue_idr  roas_net  cac_idr  tier  rank_overall  rank_in_channel  spend_share_pct
    CMP-003       GGL | Search - Brand | Multi-product | Jan Google Ads   Search - Brand   35749800        341606000      9.56  66081.0 Scale             1                1              2.5
    CMP-012         META | Retargeting | Multi-product | Mar   Meta Ads      Retargeting   35651700        252420000      7.08 143757.0 Scale             2                1              2.5
    CMP-027         META | Retargeting | Multi-product | May   Meta Ads      Retargeting   24406800        149556000      6.13 160571.0 Scale             3                2              1.7
    CMP-018       GGL | Search - Brand | Multi-product | Apr Google Ads   Search - Brand   26707800        159851000      5.99  93712.0 Scale             4                2              1.9
    CMP-013    TTK | Live Shopping | Brightening S

The best results come from two kinds of campaign that reach people who already know the brand: Google brand search (9.6x and 6.0x) and Meta retargeting (7.1x and 6.1x). Each holds under 3% of spend, and their ROAS probably overstates what extra budget would buy, because they collect demand that other campaigns created.

At the bottom sit all six awareness campaigns on Meta and TikTok, all three macro-influencer campaigns, the celebrity campaign and both display campaigns. Two TikTok conversion campaigns and one generic search campaign also fall below 1.0x.

## 3. Where the budget sits

`sql/04_budget_by_tier.sql`: spend and return by tier.

In [4]:
tiers = run("04_budget_by_tier.sql", export="budget_by_tier")

 tier  campaigns  spend_idr  spend_share_pct  net_revenue_idr  roas_net  spend_minus_net_revenue_idr
Scale          7  184581600             13.0       1103519000      5.98                   -918937400
 Keep         12  397563100             28.0        870477000      2.19                   -472913900
  Fix          6  186748900             13.2        229445000      1.23                    -42696100
  Cut         15  648595900             45.8        317592000      0.49                    331003900

Fifteen of 40 campaigns are in the Cut tier. They hold 45.8% of spend and return 0.49x. The seven Scale campaigns hold 13.0% of spend and return 5.98x.

For the Cut tier, spend minus net revenue is about IDR 331 million. That is an upper bound on the cash lost, because product cost is not in the data.

## 4. Monthly trend

`sql/05_monthly_trend.sql`: spend, CAC, CPM, conversion rate and net ROAS by month, with month-on-month change in CAC.

In [5]:
monthly = run("05_monthly_trend.sql", export="monthly_trend")

  month  spend_idr  new_customers  cac_idr  cac_change_pct  cpm_idr  cvr_pct  roas_net
2026-01   29421400            237 124141.0             NaN  57661.0     2.71      3.32
2026-02   90736300            503 180390.0            45.3  41912.0     1.88      1.97
2026-03  243499800           1012 240612.0            33.4  32693.0     1.40      1.40
2026-04  183793400           1195 153802.0           -36.1  29651.0     1.91      2.32
2026-05  217965000           1306 166895.0             8.5  30592.0     1.80      2.17
2026-06  251613600           1151 218604.0            31.0  30704.0     1.58      1.81
2026-07  208640800            962 216882.0            -0.8  29454.0     1.25      1.51
2026-08  149620900            483 309774.0            42.8  29542.0     1.21      1.14
2026-09   42198300            184 229339.0           -26.0  27884.0     1.41      1.48

Monthly CAC jumps around because the mix of campaigns changes from month to month, so I do not read much into single months. The steadier signals are conversion rate and net ROAS. Conversion rate falls from 2.71% in January to 1.21% in August, and net ROAS falls from 2.32 in April to 1.14 in August. CPM is lower from March onward than in January and February, so the media price is not what hurts. Conversion is. September has little spend and only 184 new customers, so I leave it out of the reading.

## 5. Quarterly CAC by channel

`sql/06_quarterly_cac_by_channel.sql`: is the rise in CAC broad or concentrated?

In [6]:
quarterly = run("06_quarterly_cac_by_channel.sql", export="quarterly_cac_by_channel")

   channel quarter  spend_idr  new_customers  cac_idr  cac_change_pct
Google Ads      Q1   92810500            761 121959.0             NaN
Google Ads      Q2  165067000           1044 158110.0            29.6
Google Ads      Q3   59380500            264 224926.0            42.3
Influencer      Q1  143955100            362 397666.0             NaN
Influencer      Q2   56389000            187 301545.0           -24.2
Influencer      Q3   76495600            268 285431.0            -5.3
  Meta Ads      Q1   58402400            306 190858.0             NaN
  Meta Ads      Q2  285055800           1666 171102.0           -10.4
  Meta Ads      Q3  127264000            577 220562.0            28.9
TikTok Ads      Q1   68489500            323 212042.0             NaN
TikTok Ads      Q2  146860200            755 194517.0            -8.3
TikTok Ads      Q3  137319900            520 264077.0            35.8

From Q2 to Q3, CAC rises 42% for Google, 36% for TikTok and 29% for Meta, so the increase is broad. Influencer CAC falls 5%, but its Q1 figure is inflated by the celebrity campaign (IDR 84 million of spend, CAC near IDR 649,000).

## 6. Customer lifetime value

`sql/07_customer_ltv.sql`: customers acquired from January to April, so each has at least five months to buy again. LTV is Completed-order revenue per customer.

In [7]:
ltv = run("07_customer_ltv.sql", export="customer_ltv")

   channel  customers  orders_per_customer  repeat_rate_pct  ltv_idr  cac_idr  ltv_to_cac
Google Ads       1106                 2.48             77.1 704125.0 134560.0        5.23
  Meta Ads        821                 2.14             69.7 588886.0 172845.0        3.41
TikTok Ads        604                 1.96             63.7 466051.0 171445.0        2.72
Influencer        416                 2.57             77.4 688125.0 368195.0        1.87

Counting repeat purchases changes the picture for Influencer. Its customers repeat as often as Google's (77%) and its LTV of about IDR 688,000 is close to Google's IDR 704,000, but its CAC is about 2.7 times higher, so its LTV to CAC ratio is the lowest at 1.9. TikTok customers have the lowest LTV (IDR 466,000) and the lowest repeat rate (64%).

These ratios use revenue. With a product margin below 100%, the break-even ratio is well above 1.

## 7. Cancellations and payment method

`sql/08_cancellations_by_payment.sql`: how much attributed revenue is lost to cancelled and returned orders.

In [8]:
cancellations = run("08_cancellations_by_payment.sql", export="cancellations_by_payment")

   channel  payment_method  orders  cancelled_pct  returned_pct  lost_revenue_idr
Google Ads             COD     255           12.9           2.4          13339000
Google Ads            QRIS     635            3.3           2.4          12009000
Google Ads        E-Wallet     665            2.9           1.7           9993000
Google Ads     Credit Card     552            2.7           3.1          10043000
Google Ads Virtual Account     804            2.6           2.7          12680000
Influencer             COD     204           15.2           1.5           9291000
Influencer        E-Wallet     245            4.1           2.0           3962000
Influencer     Credit Card      83            2.4           4.8           1468000
... 4 more rows, full table in data/analytics/cancellations_by_payment.csv ...
   channel  payment_method  orders  cancelled_pct  returned_pct  lost_revenue_idr
  Meta Ads        E-Wallet     894            3.1           2.2          14551000
  Meta Ads Virtual 

Cash on delivery cancels at 12% to 16% of orders. Every other payment method stays between 1% and about 4%. TikTok is the most exposed: 35% of its orders are COD (603 of 1,739) and 16.3% of those are cancelled. Returns vary less by payment method, between about 1.5% and 5%.

## 8. Calendar effects

`sql/09_calendar_effects.sql`: conversion rate by type of day.

In [9]:
calendar = run("09_calendar_effects.sql", export="calendar_effects")

       day_type  campaign_days  cvr_pct  avg_spend_per_campaign_day  avg_conversions_per_campaign_day
    Double date             78     2.47                    701447.0                              7.60
  Payday window            660     1.70                    545405.0                              4.13
Regular weekday           1240     1.44                    584917.0                              3.39
        Weekend            474     1.58                    585470.0                              3.80

Double dates (1.1, 2.2 and so on up to 9.9) convert at 2.47%, against 1.44% on regular weekdays. The payday window (25th to the 2nd) converts at 1.70% and weekends at 1.58%. Spend per campaign-day on double dates is only about 20% above a regular day, while conversions per campaign-day are 2.2 times higher. Budgets do not follow the demand peaks. There are only 78 double-date campaign-days, so this is a pattern to test rather than a settled number.

## 9. Anomaly checks

Two checks for days that look like account problems instead of real performance.

`sql/10_anomaly_checks.sql` finds days where spend is at least 3 times the campaign's own average.

In [10]:
spend_spikes = run("10_anomaly_checks.sql")

campaign_id                             campaign_name       date  spend_idr  spend_vs_campaign_avg  conversions
    CMP-020 META | Conversion | Glow Kit Bundle | Apr 2026-05-14    6239500                    7.2            9

One day stands out. On 14 May 2026 campaign CMP-020 spent IDR 6.2 million, 7.2 times its own average, and produced 9 conversions. That is about IDR 5.4 million above a normal day for this campaign, and it looks like a budget or bid setup error.

`sql/11_zero_conversion_streaks.sql` finds runs of four or more days with spend and no conversions, in campaigns that normally convert every day.

In [11]:
zero_streaks = run("11_zero_conversion_streaks.sql")

campaign_id                            campaign_name  first_day   last_day  days  spend_idr  normal_daily_conversions
    CMP-033 TTK | Conversion | Glow Kit Bundle | Jun 2026-07-08 2026-07-11     4    2032500                       2.5

Campaign CMP-033 spent IDR 2.0 million from 8 to 11 July with zero conversions, against about 2.5 a day before. Spend was normal while conversions stopped, which is the pattern of a broken tracking pixel. The platform revenue for those days is zero, so any ROAS calculated on them is understated.

## 10. Do the repaired rows change the answer?

`sql/12_flag_sensitivity.sql`: channel net ROAS on all rows against net ROAS with every campaign-day that contains a repaired row removed from both spend and revenue.

In [12]:
sensitivity = run("12_flag_sensitivity.sql", export="flag_sensitivity")

   channel  roas_net_all_rows  roas_net_without_repaired_days  difference_pct  share_of_spend_kept_pct
Google Ads               2.81                            2.67            -4.8                     90.7
Influencer               0.81                            0.79            -1.8                     91.2
  Meta Ads               2.29                            2.18            -4.8                     85.4
TikTok Ads               0.94                            0.92            -2.0                     89.0

Removing the repaired campaign-days lowers net ROAS by 2% to 5% in each channel, and the order of the channels does not change. The repairs from notebook 01 do not drive the conclusions above, although the comparison drops 9% to 15% of spend, so it is a check and not a second estimate.

## What comes next

Notebook 03 will test the main findings in Python: whether the gap between channels is larger than random variation, how much budget could move from the Cut tier to the Scale and Keep tiers, and what that is worth under stated assumptions about diminishing returns. The result tables saved in `data/analytics/` feed the Looker Studio dashboard.